In [1]:
# 1. Leia o arquivo 'videos-preparados.snappy.parquet' no dataframe 'df_video'

from pyspark.sql import SparkSession
import gdown

# iniciar spark
spark = SparkSession.builder.appName("colab").getOrCreate()

# baixar arquivo do Google Drive
file_id = "1T6Eq1UfiAAj5TW8GwdYPMU7JkOq3eUAU"
gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "videos-preparados.snappy.parquet",
    quiet=False
)

# ler arquivo parquet
df_video = spark.read.parquet("/content/videos-preparados.snappy.parquet")

# mostrar 3 registros
df_video.show(3)

Downloading...
From: https://drive.google.com/uc?id=1T6Eq1UfiAAj5TW8GwdYPMU7JkOq3eUAU
To: /content/videos-preparados.snappy.parquet
100%|██████████| 246k/246k [00:00<00:00, 68.7MB/s]


+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|    4|         30.0|[0.6985786560867407]|[0.02303716158264...|[378858.0,1.79752...|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|   news|  6379|    4853|  808787|     820019|2022|    8|         37.0|[0.8936407990235931]|[3.87946679100418...|[6379.0,808787.0,...|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|   news|  1029|    2347|   97434|     100810|202

In [2]:
# 2.Leia o arquivo ‘video-comments-tratados.snappy.parquet' no dataframe 'df_comments'

from pyspark.sql import SparkSession
import gdown

# iniciar spark
spark = SparkSession.builder.appName("colab").getOrCreate()

# baixar arquivo do Google Drive
file_id = "1UQ0xTApVeLsuL17__ozKRmUtOMI1PsFJ"
gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "video-comments-tratados.snappy.parquet",
    quiet=False
)

# ler arquivo parquet
df_comments = spark.read.parquet("/content/video-comments-tratados.snappy.parquet")

# mostrar 3 registros
df_comments.show(3)

Downloading...
From: https://drive.google.com/uc?id=1UQ0xTApVeLsuL17__ozKRmUtOMI1PsFJ
To: /content/video-comments-tratados.snappy.parquet
100%|██████████| 1.99M/1.99M [00:00<00:00, 147MB/s]


+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments| Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|Let's not forget ...|        1|           95|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|Here in NZ 50% of...|        0|           19|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|I will forever ac...|        2|          161|
+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
only showing top 3 r

In [5]:
# 3. Crie tabelas temporárias para ambos os dataframe

df_video.createOrReplaceTempView("videos")
df_comments.show(3)

df_comments.createOrReplaceTempView("comments")
df_comments.show(3)

+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments| Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|Let's not forget ...|        1|           95|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|Here in NZ 50% of...|        0|           19|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|I will forever ac...|        2|          161|
+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
only showing top 3 r

In [7]:
# 4. Faça um join das tabelas utilizando spark.sql no dataframe ‘join_video_comments’

join_video_comments = spark.sql("""
SELECT *
FROM videos v
JOIN comments c
ON v.`Video ID` = c.`Video ID`
""")

join_video_comments.show(3)

+--------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments| Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|Keyword|Likes|Comments| Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   te

In [8]:
# 5. Faça as mesmas etapas anteriores (1,2,3,4) utilizando repartition e coalesce

# Repartition (aumenta/distribui melhor)
df_video_rep = df_video.repartition(4)
df_comments_rep = df_comments.repartition(4)

# Coalesce (reduz número de partições)
df_video_final = df_video_rep.coalesce(2)
df_comments_final = df_comments_rep.coalesce(2)

# Criar tabelas temporárias
df_video_final.createOrReplaceTempView("videos_rep")
df_comments_final.createOrReplaceTempView("comments_rep")

# Join com SQL
join_video_comments = spark.sql("""
SELECT *
FROM videos_rep v
JOIN comments_rep c
ON v.`Video ID` = c.`Video ID`
""")

join_video_comments.show(3)

+--------------------+-----------+------------+----------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|   Keyword|Likes|Comments| Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|   Keyword|Likes|Comments| Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+----------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|The Literature Re...|jKL2pdRmwc4|

In [10]:
# 6. Utilize o explain e refaça as etapas (1,2,3,4) com join otimizado e filter apenas com dados necessários

from pyspark.sql import SparkSession
import gdown

# iniciar spark
spark = SparkSession.builder.appName("colab").getOrCreate()

# =========================
# 1. Baixar arquivos
# =========================

# videos
gdown.download(
    "https://drive.google.com/uc?id=1T6Eq1UfiAAj5TW8GwdYPMU7JkOq3eUAU",
    "videos-preparados.snappy.parquet",
    quiet=False
)

# comments
gdown.download(
    "https://drive.google.com/uc?id=1UQ0xTApVeLsuL17__ozKRmUtOMI1PsFJ",
    "video-comments-tratados.snappy.parquet",
    quiet=False
)

# =========================
# 2. Ler arquivos
# =========================

df_video = spark.read.parquet("/content/videos-preparados.snappy.parquet")
df_comments = spark.read.parquet("/content/video-comments-tratados.snappy.parquet")

# =========================
# 3. Selecionar colunas necessárias (otimização)
# =========================

df_video_sel = df_video.select("Video ID", "Keyword", "Views", "Likes")
df_comments_sel = df_comments.select("Video ID", "Comment", "Sentiment")

# =========================
# 4. Criar tabelas temporárias
# =========================

df_video_sel.createOrReplaceTempView("videos_opt")
df_comments_sel.createOrReplaceTempView("comments_opt")

# =========================
# 5. Join + filtro otimizado
# =========================

join_video_comments = spark.sql("""
SELECT
    v.`Video ID`,
    v.Keyword,
    v.Views,
    v.Likes,
    c.Comment,
    c.Sentiment
FROM videos_opt v
JOIN comments_opt c
ON v.`Video ID` = c.`Video ID`
WHERE c.Sentiment = 1
""")

# =========================
# 6. Mostrar resultado
# =========================

join_video_comments.show(3)

# =========================
# 7. Explain (plano de execução)
# =========================

join_video_comments.explain(True)


Downloading...
From: https://drive.google.com/uc?id=1T6Eq1UfiAAj5TW8GwdYPMU7JkOq3eUAU
To: /content/videos-preparados.snappy.parquet
100%|██████████| 246k/246k [00:00<00:00, 79.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1UQ0xTApVeLsuL17__ozKRmUtOMI1PsFJ
To: /content/video-comments-tratados.snappy.parquet
100%|██████████| 1.99M/1.99M [00:00<00:00, 171MB/s]


+-----------+-------+------+-----+--------------------+---------+
|   Video ID|Keyword| Views|Likes|             Comment|Sentiment|
+-----------+-------+------+-----+--------------------+---------+
|wAZZ-UWGVHI|   tech|135612| 3407|Let's not forget ...|        1|
|wAZZ-UWGVHI|   tech|135612| 3407|We’ve been houndi...|        1|
|wAZZ-UWGVHI|   tech|135612| 3407|For now, I need b...|        1|
+-----------+-------+------+-----+--------------------+---------+
only showing top 3 rows
== Parsed Logical Plan ==
'Project ['v.Video ID, 'v.Keyword, 'v.Views, 'v.Likes, 'c.Comment, 'c.Sentiment]
+- 'Filter ('c.Sentiment = 1)
   +- 'Join Inner, ('v.Video ID = 'c.Video ID)
      :- 'SubqueryAlias v
      :  +- 'UnresolvedRelation [videos_opt], [], false
      +- 'SubqueryAlias c
         +- 'UnresolvedRelation [comments_opt], [], false

== Analyzed Logical Plan ==
Video ID: string, Keyword: string, Views: int, Likes: int, Comment: string, Sentiment: int
Project [Video ID#347, Keyword#349, Views#35

In [11]:
# 7. Salve o seu join otimizado como 'join-videos-comments-otimizado' no formato parquet

join_video_comments.write.parquet(
    "/content/join-videos-comments-otimizado",
    mode="overwrite"
)

# =========================
# Utilize o explain e refaça as etapas com otimização
# =========================

from pyspark.sql import SparkSession
import gdown

# Inicia a sessão do Spark (necessária para qualquer operação)
spark = SparkSession.builder.appName("colab").getOrCreate()

# -------------------------
# 1. Download dos arquivos
# -------------------------

# Baixa o dataset de vídeos do Google Drive
gdown.download(
    "https://drive.google.com/uc?id=1T6Eq1UfiAAj5TW8GwdYPMU7JkOq3eUAU",
    "videos-preparados.snappy.parquet",
    quiet=False
)

# Baixa o dataset de comentários do Google Drive
gdown.download(
    "https://drive.google.com/uc?id=1UQ0xTApVeLsuL17__ozKRmUtOMI1PsFJ",
    "video-comments-tratados.snappy.parquet",
    quiet=False
)

# -------------------------
# 2. Leitura dos arquivos
# -------------------------

# Lê o arquivo parquet de vídeos em um DataFrame
df_video = spark.read.parquet("/content/videos-preparados.snappy.parquet")

# Lê o arquivo parquet de comentários em um DataFrame
df_comments = spark.read.parquet("/content/video-comments-tratados.snappy.parquet")

# -------------------------
# 3. Seleção de colunas (otimização)
# -------------------------

# Seleciona apenas colunas necessárias do dataset de vídeos
# Isso reduz o volume de dados processados (melhora performance)
df_video_sel = df_video.select("Video ID", "Keyword", "Views", "Likes")

# Seleciona apenas colunas necessárias do dataset de comentários
df_comments_sel = df_comments.select("Video ID", "Comment", "Sentiment")

# -------------------------
# 4. Criação de tabelas temporárias
# -------------------------

# Cria uma view temporária para permitir uso de SQL no Spark
df_video_sel.createOrReplaceTempView("videos_opt")

# Cria outra view para os comentários
df_comments_sel.createOrReplaceTempView("comments_opt")

# -------------------------
# 5. Join otimizado + filtro
# -------------------------

# Realiza o join entre vídeos e comentários
# Usa crase (`) pois o nome da coluna possui espaço
# Aplica filtro para reduzir dados (ex: apenas sentimentos positivos)
join_video_comments = spark.sql("""
SELECT
    v.`Video ID`,
    v.Keyword,
    v.Views,
    v.Likes,
    c.Comment,
    c.Sentiment
FROM videos_opt v
JOIN comments_opt c
ON v.`Video ID` = c.`Video ID`
WHERE c.Sentiment = 1
""")

# Mostra os primeiros registros do resultado
join_video_comments.show(3)

# -------------------------
# 6. Explain (plano de execução)
# -------------------------

# Exibe o plano lógico e físico da query
# Permite entender como o Spark executa o processamento
join_video_comments.explain(True)


# =========================
# 7. Salvar resultado em parquet
# =========================

# Salva o DataFrame resultante do join em formato parquet
# O modo "overwrite" evita erro caso o diretório já exista
join_video_comments.write.parquet(
    "/content/join-videos-comments-otimizado",
    mode="overwrite"
)